# How to Fine-Tune a Decision Model with TRL (SFT + LoRA)


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/finetuning/notebooks/sft_for_decision_models_with_trl.ipynb)

This tutorial shows you how to fine-tune a decision model, a model that makes decisions instead of generating text, on your own labels.

We fine-tune our [`d1-3B`](https://huggingface.co/LiquidAI/d1-3B) decision model with Supervised Fine-tuning (SFT) and LoRA using the TRL library, with intent classification for customer support routing on Banking77 as the example.

Follow along if it's your first time using trl, or take single code snippets for your own workflow

## 🎯 What You'll Find:
- **LoRA + SFT** - Fine-tune the d1-3B decision model on your own decisions using LoRA (from PEFT)
- **Evaluation** - Compare the decision model's accuracy before and after fine-tuning

## 📋 Prerequisites:
- **GPU Runtime**: Select a GPU in `Runtime` → `Change runtime type`. A free T4 GPU is enough (about 35 minutes of training)
- **Hugging Face Account**: Only needed to push your model to the Hub

# 📦 Installation & Setup

First, let's install all the required packages:

In [1]:
!pip install -qqq "transformers==5.19.0" "trl==1.15.0" "peft==0.21.2" "torchao>0.16.0" datasets --progress-bar off

Let's now verify the packages are installed correctly

In [2]:
import torch
import transformers
import trl
import peft
import os
os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"📊 TRL version: {trl.__version__}")
print(f"🎛️ PEFT version: {peft.__version__}")

# bfloat16 on GPUs that support it natively (e.g. A100), float16 otherwise (e.g. T4)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
print(f"🔢 Precision: {dtype}")

📦 PyTorch version: 2.11.0+cu130
🤗 Transformers version: 5.19.0
📊 TRL version: 1.15.0
🎛️ PEFT version: 0.21.2
🔢 Precision: torch.bfloat16


# Loading the model from Transformers 🤗

d1-3B ships its own code for the System One API (`system_one` and `system_one_batch`), so we load it with `trust_remote_code=True`.

We also download the model repository locally. It contains `prompt.py`, the module d1-3B uses to turn a state and a question into a prompt. We will reuse it to build our training data, so that the model sees the exact same prompt during training and inference.

In [3]:
import sys
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer

model_id = "LiquidAI/d1-3B"

# Download the checkpoint and d1's own code, and make its prompt builder importable
local_dir = snapshot_download(model_id)
sys.path.insert(0, local_dir)

print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("🧠 Loading model...")
model = AutoModel.from_pretrained(
    model_id,
    trust_remote_code=True,
    dtype=dtype,
    ).to("cuda")

print("✅ Model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")
print(f"📖 Vocab size: {len(tokenizer)}")
print(f"💾 Model size: ~{model.num_parameters() * 2 / 1e9:.1f} GB ({dtype})")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

📚 Loading tokenizer...
🧠 Loading model...


[transformers] A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- prompt.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- hybrid.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- api.py
- lfm2_vl.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] A new version of the following files was downloaded from https://huggingface.co/LiquidAI/d1-3B:
- runner.py
. Make sure to double-check they do not contain 

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

✅ Model loaded successfully!
🔢 Parameters: 3,123,483,888
📖 Vocab size: 125017
💾 Model size: ~6.2 GB (torch.bfloat16)


# 💡 What is a decision model?

A decision model is a model that makes decisions instead of generating text: it answers yes/no, multiple-choice, and scoring questions with a probability for each answer, in a single forward pass. Typical uses are classification, routing and triage, content moderation, and LLM-as-a-judge scoring.

d1-3B is a 3B parameter decision model built on [LFM2.5-VL-3B](https://huggingface.co/LiquidAI/LFM2.5-VL-3B). It takes a state and a set of questions, and returns typed answers in one forward pass, without generating any output tokens. Each question uses one of the following three primitives:

- `noul`: Yes or no, as P(yes)
- `choice`: One of named options, with a probability for each
- `score`: A level on an ordered scale of 2 to 10 levels

Before we fine-tune, let's see how d1-3B makes a decision. We ask a `choice` question about a customer message:

In [4]:
state = "I was charged twice this month, please refund one of them."

questions = {
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this?",
        "criteria": {
            "billing": "Charges, refunds, invoices",
            "technical": "App or site faults",
            "fraud": "Suspected unauthorised use",
        },
    },
}

response = model.system_one(state=state, questions=questions)

print(response)

{'answers': {'team': {'type': 'choice', 'choice': 'billing', 'confidence': 0.982860184342666, 'probabilities': {'billing': 0.982860184342666, 'technical': 0.01091859038543887, 'fraud': 0.0062212252718952575}}}, 'usage': {'input_tokens': 61, 'output_tokens': 0}}


Under the hood, d1-3B renders the state and the question into a single prompt. Each option gets a short code (`A`, `B`, `C`, ...), and the prompt ends right after the assistant header:

In [5]:
from prompt import aliases, as_question, render

rendered_prompt = render(
    tokenizer=tokenizer,
    state=state,
    q=as_question(questions["team"]),
    bos=tokenizer.bos_token,
)

print(rendered_prompt)

<|startoftext|><|im_start|>user
I was charged twice this month, please refund one of them.


QUESTION:
Which team should handle this?

Options:
A Charges, refunds, invoices
B App or site faults
C Suspected unauthorised use

Reply with the option code only.<|im_end|>
<|im_start|>assistant



The model then runs one forward pass and reads the next-token probabilities at the end of the prompt, the **answer slot**. It only looks at the tokens of the option codes and normalizes them into the probabilities you saw above.

This means that fine-tuning a decision model is regular SFT where **the completion is a single token: the code of the correct option**, followed by the end-of-turn token `<|im_end|>`.

# 🎯 Supervised Fine-Tuning (SFT + LoRA)

SFT teaches the model to make your decisions by training on input-output pairs. For a decision model, the input is the rendered prompt and the output is the code of the correct option.

We use LoRA (Low-Rank Adaptation) to finetune the model by only training a small number of additional parameters.

## Load an SFT Dataset

We will use [Banking77](https://huggingface.co/datasets/mteb/banking77), a dataset of ~13k online banking customer messages, each labeled with one of 77 fine-grained intents. To keep this notebook fast on a free T4 GPU, we train on 3,000 messages and evaluate on 100. Feel free to increase these numbers with `select(range(...))`.

In [6]:
from datasets import load_dataset

print("📥 Loading dataset...")
dataset = load_dataset("mteb/banking77")
train_dataset = dataset["train"].shuffle(seed=42).select(range(3000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(100))

print("✅ Dataset loaded:")
print(f"   📚 Train samples: {len(train_dataset)}")
print(f"   🧪 Test samples: {len(test_dataset)}")
print(f"\n📝 Single Sample: {train_dataset[0]}")

📥 Loading dataset...


README.md:   0%|          | 0.00/16.6k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  294kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 91.4kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9993 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3076 [00:00<?, ? examples/s]

✅ Dataset loaded:
   📚 Train samples: 3000
   🧪 Test samples: 100

📝 Single Sample: {'text': 'How long is a top up supposed to be pending, because an hour is too long! Can you fix this to go through?', 'label': 47, 'label_text': 'pending_top_up'}


## Data processing

Banking77 contains reworded versions of the same request, such as "can you tell me why my transfer failed?" and "Can you please tell me why my transfer failed?". If one lands in training and the other in evaluation, the evaluation rewards memorization and overstates what fine-tuning gained.

So we split by **case**: messages that ask the same thing stay on the same side. If your own data has a case, ticket, or conversation ID, group by that. Banking77 has none, so we approximate one: two messages belong to the same case when they share at least 80% of their words.

In [7]:
import random
import re
from collections import Counter

def words(text):
    return set(re.findall(r"[a-z0-9']+", text.lower()))

def same_case(a, b, threshold=0.8):
    """Whether two messages' word sets overlap enough to count as rewordings of one request."""
    union = a | b
    return bool(union) and len(a & b) / len(union) >= threshold

def case_ids(texts):
    """Give messages that reword each other the same case ID, transitively."""
    word_sets = [words(text) for text in texts]
    parent = list(range(len(texts)))

    def find(i):
        while parent[i] != i:
            i = parent[i]
        return i

    for i in range(len(texts)):
        for j in range(i + 1, len(texts)):
            if same_case(word_sets[i], word_sets[j]):
                parent[find(j)] = find(i)
    return [find(i) for i in range(len(texts))]

# 1. Drop training messages that reword a test message
test_words = [words(text) for text in test_dataset["text"]]
leaked = {i for i, text in enumerate(train_dataset["text"]) if any(same_case(words(text), w) for w in test_words)}
train_dataset = train_dataset.select([i for i in range(len(train_dataset)) if i not in leaked])

# 2. Hold out 10% of the training messages to track the loss, taking whole cases only
cases = case_ids(train_dataset["text"])
case_sizes = Counter(cases)
shuffled_cases = sorted(case_sizes)
random.Random(42).shuffle(shuffled_cases)
eval_cases, n_eval = set(), 0
for case in shuffled_cases:
    if n_eval >= 0.1 * len(train_dataset):
        break
    eval_cases.add(case)
    n_eval += case_sizes[case]

sft_train = train_dataset.select([i for i, case in enumerate(cases) if case not in eval_cases])
sft_eval = train_dataset.select([i for i, case in enumerate(cases) if case in eval_cases])

print(f"🧹 Dropped {len(leaked)} training messages that reword a test message")
print(f"🔗 {len(cases) - len(case_sizes)} training messages reword another training message")
print(f"✅ No case on both sides: {len(sft_train)} train and {len(sft_eval)} eval messages")

🧹 Dropped 6 training messages that reword a test message
🔗 77 training messages reword another training message
✅ No case on both sides: 2694 train and 300 eval messages


## Define the decision

We turn the classification task into a single `choice` question. Each intent becomes one option, described by its label name.

**Important**: The option codes are assigned by position: the first option gets the first code, the second option the second, and so on. If we always trained with the options in the same order, the model could learn which code means which intent without reading the option list. Adding or reordering an intent later would then shift the codes and break the fine-tuned model. We avoid that below by giving every training example its own random subset of the intents, in a random order.

In [8]:
labels = sorted(set(dataset["train"]["label_text"]))

intent_question = {
    "type": "choice",
    "instructions": "Which intent does this banking customer message express?",
    "criteria": {label: label.replace("_", " ") for label in labels},
}

# The code d1-3B assigns to each option in this order, e.g. "card_arrival" -> "12"
codes = {label: code for label, (code, _) in zip(labels, aliases(tokenizer, labels))}

print(f"🏷️ Number of options: {len(labels)}")
print(f"🔤 First option codes: {dict(list(codes.items())[:5])}")

🏷️ Number of options: 77
🔤 First option codes: {'Refund_not_showing_up': '00', 'activate_my_card': '01', 'age_limit': '02', 'apple_pay_or_google_pay': '03', 'atm_support': '04'}


With up to 26 options, d1-3B uses the letters `A` to `Z` as codes. With more options, as here, it switches to the two-digit codes `00` to `76`, each of which is a single token. Going from 26 to 27 options therefore changes every code at once, which is another reason to train on shuffled option lists. Our subsets keep at least 27 options, so the codes stay two-digit, as they are when we ask with all 77 intents.

We now transform the dataset to a format compatible with `trl`. We use TRL's **prompt-completion** format:
- `prompt`: the full prompt rendered by d1-3B's own prompt builder, with its own option list: the correct intent plus a random number of the other intents (27 to 77 options in total), in a random order
- `completion`: the code the correct intent gets in that list, followed by `<|im_end|>`

Because the list changes from example to example, a code says nothing about which intent it stands for: the model has to read the option list to find the right one, whichever intents it contains.

With a prompt-completion dataset, `SFTTrainer` computes the loss only on the completion, so the model learns to put its probability mass on the right option code.

In [9]:
def to_prompt_completion(example, idx, split):
    # Every example gets its own option list: the correct intent plus 26 to 76 others, in a random order
    rng = random.Random(f"{split}-{idx}")
    others = [label for label in labels if label != example["label_text"]]
    options = rng.sample(others, rng.randint(26, len(others))) + [example["label_text"]]
    rng.shuffle(options)
    question = as_question({**intent_question, "criteria": {label: intent_question["criteria"][label] for label in options}})
    option_codes = {label: code for label, (code, _) in zip(options, aliases(tokenizer, options))}
    return {
        "prompt": render(tokenizer, example["text"], question, bos=tokenizer.bos_token),
        "completion": option_codes[example["label_text"]] + tokenizer.eos_token,
    }

# Each example gets one option list, which is enough for one epoch. To train for more epochs, map the dataset
# once per epoch with a different `split` value and concatenate the copies, so every pass sees new lists.
train_dataset_sft = sft_train.map(to_prompt_completion, with_indices=True, fn_kwargs={"split": "train"}, remove_columns=sft_train.column_names)
eval_dataset_sft = sft_eval.map(to_prompt_completion, with_indices=True, fn_kwargs={"split": "eval"}, remove_columns=sft_eval.column_names)

print("✅ SFT Dataset formatted:")
print(f"   📚 Train samples: {len(train_dataset_sft)}")
print(f"   🧪 Eval samples: {len(eval_dataset_sft)}")
print(f"\n📝 Prompt:\n{train_dataset_sft[0]['prompt']}")
print(f"\n🎯 Completion: {train_dataset_sft[0]['completion']!r}")

Map:   0%|          | 0/2694 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

✅ SFT Dataset formatted:
   📚 Train samples: 2694
   🧪 Eval samples: 300

📝 Prompt:
<|startoftext|><|im_start|>user
Whats the process of tranfering money into an account?


QUESTION:
Which intent does this banking customer message express?

Options:
00 reverted card payment?
01 card acceptance
02 cash withdrawal charge
03 card swallowed
04 declined cash withdrawal
05 get physical card
06 wrong exchange rate for cash withdrawal
07 request refund
08 wrong amount of cash received
09 automatic top up
10 transaction charged twice
11 apple pay or google pay
12 top up by bank transfer charge
13 edit personal details
14 card payment not recognised
15 card linking
16 verify my identity
17 supported cards and currencies
18 extra charge on statement
19 compromised card
20 age limit
21 terminate account
22 top up by card charge
23 card delivery estimate
24 visa or mastercard
25 direct debit payment not recognised
26 atm support
27 order physical card
28 cash withdrawal not recognised
29 exchange c

Let's look at two training messages with the same intent. Each prompt has its own option list, so `card arrival` gets a different code in each, and the model has to read the list to answer.

In [10]:
import re

# The first two training messages whose intent is card_arrival
for i in [i for i, label in enumerate(sft_train["label_text"]) if label == "card_arrival"][:2]:
    options = re.findall(r"^\d\d .+$", train_dataset_sft[i]["prompt"], re.M)
    print(f"💬 Message: {sft_train[i]['text']}")
    print(f"🔢 Number of options: {len(options)}")
    print(f"📋 First options: {options[:3]}")
    print(f"🔎 Correct option: {next(option for option in options if option.endswith(' card arrival'))}")
    print(f"🎯 Completion: {train_dataset_sft[i]['completion']!r}\n")

💬 Message: Where is the card I ordered 2 weeks ago?
🔢 Number of options: 76
📋 First options: ['00 get disposable virtual card', '01 edit personal details', '02 getting virtual card']
🔎 Correct option: 08 card arrival
🎯 Completion: '08<|im_end|>'

💬 Message: Can I track when my card will be delivered?
🔢 Number of options: 69
📋 First options: ['00 activate my card', '01 getting spare card', '02 getting virtual card']
🔎 Correct option: 55 card arrival
🎯 Completion: '55<|im_end|>'



## Evaluate the base model

Let's measure how well d1-3B performs on the 100 test messages before fine-tuning. `system_one_batch` packs many messages into each forward pass.

In [11]:
from tqdm.auto import tqdm

def evaluate(model, dataset, batch_size=16):
    """Accuracy of the model's intent decisions on `dataset`, and its predictions."""
    predictions = []
    # Evaluate in batches, so the progress bar shows how far we are
    for start in tqdm(range(0, len(dataset), batch_size), desc="Evaluating"):
        texts = dataset[start:start + batch_size]["text"]
        responses = model.system_one_batch([(text, {"intent": intent_question}) for text in texts])
        predictions += [response["answers"]["intent"]["choice"] for response in responses]
    accuracy = sum(p == y for p, y in zip(predictions, dataset["label_text"])) / len(dataset)
    return accuracy, predictions

print("📏 Evaluating the base model...")
baseline_accuracy, baseline_predictions = evaluate(model, test_dataset)
print(f"🎯 Baseline accuracy: {baseline_accuracy:.1%}")

📏 Evaluating the base model...


Evaluating:   0%|          | 0/7 [00:00<?, ?it/s]

🎯 Baseline accuracy: 83.0%


Let's free up the GPU memory before training:

In [12]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()

## Load the model for training

For training, we load the same weights with the standard Transformers class. d1-3B's checkpoint uses the standard LFM2-VL module names, so we don't need its custom code here.

In [13]:
from transformers import AutoModelForImageTextToText

print("🧠 Loading model for training...")
model = AutoModelForImageTextToText.from_pretrained(model_id, dtype=dtype, device_map="auto")
print("✅ Model loaded successfully!")

🧠 Loading model for training...


Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

✅ Model loaded successfully!


## Wrap the model with PEFT

We specify target modules that will be finetuned while the rest of the models weights remains frozen. Feel free to modify the `r` (rank) value:
- Higher → better approximation of full-finetuning
- Lower → needs even less compute resources

Our task is text-only, so we only adapt the language model: its attention layers (`q_proj`, `k_proj`, `v_proj`, `out_proj`), its short convolution layers (`in_proj`, `out_proj`), and its MLP layers (`w1`, `w2`, `w3`). The vision encoder uses some of the same layer names, so we exclude it explicitly.

In [14]:
from peft import LoraConfig, get_peft_model

target_modules = [
    "q_proj", "k_proj", "v_proj",
    "out_proj", "in_proj",
    "w1", "w2", "w3"
]

peft_config = LoraConfig(
    r=16,  # <- lower values = fewer parameters
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
    exclude_modules=r".*vision_tower.*", # The vision encoder also has q_proj, k_proj, v_proj, and out_proj layers: keep them frozen by excluding them
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

print("✅ LoRA configuration created!")
print(f"🎛️  LoRA rank: {peft_config.r}")
print(f"📊 LoRA alpha: {peft_config.lora_alpha}")

trainable params: 24,461,312 || all params: 3,147,945,200 || trainable%: 0.7771
✅ LoRA configuration created!
🎛️  LoRA rank: 16
📊 LoRA alpha: 32


## Launch Training

We are now ready to launch an SFT run with `SFTTrainer`, feel free to modify `SFTConfig` to play around with different configurations.

In [15]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="./d1-3B-banking77",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # <- effective batch size of 16
    gradient_checkpointing=True,  # <- trades compute for memory, so it fits on a T4
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,  # <- a float is a fraction of the total steps
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="no",
    bf16=dtype == torch.bfloat16,
    fp16=dtype == torch.float16,
    report_to="none",
)

print("🏗️  Creating SFT trainer...")
sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset_sft,
    eval_dataset=eval_dataset_sft,
    processing_class=tokenizer,
)

print("\n🚀 Starting SFT training...")
sft_trainer.train()

print("🎉 SFT training completed!")

sft_trainer.save_model()
print(f"💾 Saving to: {sft_config.output_dir}")

🏗️  Creating SFT trainer...


Adding EOS to train dataset:   0%|          | 0/2694 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2694 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2694 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2694 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]


🚀 Starting SFT training...


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
50,0.180615,0.226280,0.178536,0.934211,264665.000000
100,0.149379,0.170480,0.106043,0.957237,529059.000000
150,0.097676,0.149993,0.151665,0.958882,793472.000000
169,0.185679,0.149433,0.152403,0.958882,888013.000000


🎉 SFT training completed!
💾 Saving to: ./d1-3B-banking77


## Save merged model

Merge the extra weights learned with LoRA back into the model to obtain a "normal" model checkpoint.

The merged checkpoint contains only the weights and the tokenizer. To keep the System One API, we also copy d1-3B's own code and processor configuration into the same folder.

In [16]:
import shutil

output_dir = "./d1-3B-banking77-merged"

print("🔄 Merging LoRA weights...")
merged_model = sft_trainer.model.merge_and_unload()
merged_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# d1-3B's code for the System One API, its processor configuration, and its license
for filename in ["api.py", "hybrid.py", "lfm2_vl.py", "modeling_d1.py", "prompt.py", "runner.py", "processor_config.json", "LICENSE"]:
    shutil.copy(os.path.join(local_dir, filename), output_dir)

print(f"💾 Merged model saved to: {output_dir}")

# To push the merged model and d1-3B's code to the Hugging Face Hub, uncomment:
# from huggingface_hub import create_repo, upload_folder
# create_repo("your-model-path", exist_ok=True)
# upload_folder(folder_path=output_dir, repo_id="your-model-path")

🔄 Merging LoRA weights...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Merged model saved to: ./d1-3B-banking77-merged


# 📊 Evaluate the Fine-tuned Model

Let's free up the GPU memory again and load the fine-tuned model the same way we loaded d1-3B: with `AutoModel` and `trust_remote_code=True`.

In [17]:
del model, merged_model, sft_trainer
gc.collect()
torch.cuda.empty_cache()

finetuned = AutoModel.from_pretrained(output_dir, trust_remote_code=True, dtype=dtype).to("cuda")

print("📏 Evaluating the fine-tuned model...")
finetuned_accuracy, finetuned_predictions = evaluate(finetuned, test_dataset)
print(f"🎯 Baseline accuracy:   {baseline_accuracy:.1%}")
print(f"🎯 Fine-tuned accuracy: {finetuned_accuracy:.1%}")

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

📏 Evaluating the fine-tuned model...


Evaluating:   0%|          | 0/7 [00:00<?, ?it/s]

🎯 Baseline accuracy:   83.0%
🎯 Fine-tuned accuracy: 88.0%
